In [52]:
from dotenv import load_dotenv

load_dotenv()

True

In [53]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [ ]:
# this is how we define the client for a local mcp server
# basically how we want the mcp server file to be ran from the terminal

from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": { # name of the server
                "transport": "stdio", # the transport (sse, http, stdio)
                "command": "python", # the runnable (uv, python, npx)
                "args": ["resources/2.1_mcp_server.py"], # the file we want to run
            }
    }
)

In [43]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [44]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=prompt
)

In [46]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library. be very very brief. no lengthy explanations")]},
    config=config
)

In [47]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library. be very very brief. no lengthy explanations', additional_kwargs={}, response_metadata={}, id='2e32bfc9-306d-4a72-a802-c40136d4bd4c'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 280, 'total_tokens': 373, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E95EYp1rKxMohKmDqt6OHjWoqwk6w', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fcbfc-e8fc-7a72-848b-6b77cccad3ea-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': 'call_q0ZHNYpVqaQcDLkx4axLR4cf

## Online MCP

In [ ]:
# this is how we define the remote mcp server client

from langchain_mcp_adapters.client import MultiServerMCPClient
import os

github_token = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")

client = MultiServerMCPClient(
  {
  "github": {
          "transport":"stdio",
          "command": "docker",
          "args": [
        "run",
        "-i",
        "--rm",
        "-e",
        f"GITHUB_PERSONAL_ACCESS_TOKEN={github_token}",
        "mcp/github"
      ],
      "env": {
        "GITHUB_PERSONAL_ACCESS_TOKEN": github_token
      }
    }
  }
)

In [55]:
from pprint import pprint

try:
    tools = await client.get_tools()
    pprint(tools)
except Exception as e:
    pprint(repr(e))
    pprint(type(e))

[StructuredTool(name='create_or_update_file', description='Create or update a single file in a GitHub repository', args_schema={'type': 'object', 'properties': {'owner': {'type': 'string', 'description': 'Repository owner (username or organization)'}, 'repo': {'type': 'string', 'description': 'Repository name'}, 'path': {'type': 'string', 'description': 'Path where to create/update the file'}, 'content': {'type': 'string', 'description': 'Content of the file'}, 'message': {'type': 'string', 'description': 'Commit message'}, 'branch': {'type': 'string', 'description': 'Branch to create/update the file in'}, 'sha': {'type': 'string', 'description': 'SHA of the file being replaced (required when updating existing files)'}}, 'required': ['owner', 'repo', 'path', 'content', 'message', 'branch'], 'additionalProperties': False, '$schema': 'http://json-schema.org/draft-07/schema#'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_too

In [58]:
from langgraph.checkpoint.memory import InMemorySaver

system_prompt = """
help with github operations. 
Use your tools effectively. 
First analyze the query then intelligently decide which tool to use.
"""

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [59]:
question = HumanMessage(content="what's my latest repo that I pushed into?")

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [question]},
    config=config
)

pprint(response)

{'messages': [HumanMessage(content="what's my latest repo that I pushed into?", additional_kwargs={}, response_metadata={}, id='089493fb-4c0c-4431-a866-8ecd25658c35'),
              AIMessage(content='I can do that, but I need a bit more info to be precise.\n\nPlease confirm:\n- Which GitHub username (or organization) should I check? (your account)\n- Do you want:\n  a) the single repository that was most recently pushed to across all your repos (i.e., the latest push date), or\n  b) the repository that contains your most recent commit (authored by you) across all repos?\n- Should I include both public and private repos (assuming I have access to them)?\n\nIf you want, I can proceed with option a by default: find the repo under your account with the most recent push date across all repos you have access to. Just provide your GitHub username.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1883, 'prompt_tokens': 2121, 'total_tokens': 4004, 

In [60]:
question = HumanMessage(content="my github username is shamoon-ahmed. i just wanna know about the single github repo that I recetly pushed")

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [question]},
    config=config
)

pprint(response)

{'messages': [HumanMessage(content="what's my latest repo that I pushed into?", additional_kwargs={}, response_metadata={}, id='089493fb-4c0c-4431-a866-8ecd25658c35'),
              AIMessage(content='I can do that, but I need a bit more info to be precise.\n\nPlease confirm:\n- Which GitHub username (or organization) should I check? (your account)\n- Do you want:\n  a) the single repository that was most recently pushed to across all your repos (i.e., the latest push date), or\n  b) the repository that contains your most recent commit (authored by you) across all repos?\n- Should I include both public and private repos (assuming I have access to them)?\n\nIf you want, I can proceed with option a by default: find the repo under your account with the most recent push date across all repos you have access to. Just provide your GitHub username.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1883, 'prompt_tokens': 2121, 'total_tokens': 4004, 

In [61]:
pprint(response["messages"][-1].content)

('The most recently pushed repo under your account shamoon-ahmed is:\n'
 '\n'
 '- Agentic-AI-Engineering\n'
 '  - URL: https://github.com/shamoon-ahmed/Agentic-AI-Engineering\n'
 '  - Pushed at: 2026-08-02T20:15:56Z (UTC)\n'
 '\n'
 'Would you like me to pull the latest commit details or show the exact commit '
 'SHA for that push?')
